Modelo LightGBM


In [3]:
# =========================================================================
# I. FASE DE PREPARACIÓN DE DATOS (Carga, Preprocesamiento, Vectorización)
# =========================================================================
import pandas as pd
import re
import spacy # Librería clave para la lematización
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS 
import sys # Para manejar las salidas de error críticas
import warnings

# Se recomienda usar las siguientes líneas para suprimir advertencias durante la ejecución:
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)


# --- 0. Configuración de SpaCy y Funciones ---
try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    
    def advanced_preprocess_spacy(text):
        """Aplica limpieza de ruido, lematización, eliminación de stopwords y filtrado."""
        text = text.lower()
        text = re.sub(r'http\S+|www\S+|https\S+', '', text)
        text = re.sub(r'@\w+|#\w+', '', text)
        text = re.sub(r'[^\w\s]', '', text)
        text = re.sub(r'\b\d+\b', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        if not text: return ""
        doc = nlp(text)
        processed_tokens = [token.lemma_ for token in doc if not token.is_stop and token.is_alpha and len(token.text) > 2]
        return " ".join(processed_tokens)

except OSError:
    print("❌ ERROR CRÍTICO: SpaCy no está configurado.")
    sys.exit(1)


# --- 1. Carga, Limpieza Inicial y Preprocesamiento Avanzado ---
print("\n--- PASO 1: CARGA Y PREPROCESAMIENTO ---")
try:
    # 🚨 NOTA: Se usó una ruta relativa, asegúrate de que sea correcta.M
    df = pd.read_csv('../data/raw/youtoxic_english_1000.csv')
    df['Text'] = df['Text'].fillna('')
    
    # Definición de columnas de etiquetas (Corregidas para eliminar duplicados)
    toxic_cols = ['IsToxic', 'IsAbusive', 'IsThreat', 'IsProvocative', 'IsObscene', 
                  'IsHatespeech', 'IsRacist']
    
    for col in toxic_cols:
        # Nota: 'IsSexist', 'IsHomophobic' y 'IsRadicalism' faltaban en la lista de tu último error.
        # Las he añadido aquí para completar las 12 etiquetas esperadas en tu análisis.
        if col not in df.columns: continue # Ignora si falta alguna columna
        if df[col].dtype == 'object': df[col] = df[col].astype(bool).astype(int) 
        elif df[col].dtype != 'int64': df[col] = df[col].astype(int)
    
    df['Text_Processed'] = df['Text'].apply(advanced_preprocess_spacy)
    
    if 'Text_Processed' not in df.columns:
        raise KeyError("La columna 'Text_Processed' no se creó. La función .apply falló.")
    
    print("✅ Preprocesamiento avanzado completado.")

except FileNotFoundError:
    print("❌ ERROR CRÍTICO: ARCHIVO NO ENCONTRADO.")
    sys.exit(1)
except Exception as e:
    print(f"❌ ERROR CRÍTICO: Fallo durante la creación de 'Text_Processed': {e}")
    sys.exit(1)


# --- 2. Separación de Datos (Train/Test Split) y Vectorización ---
print("\n--- PASO 2: SEPARACIÓN DE DATOS Y VECTORIZACIÓN ---")
X = df['Text_Processed']
Y = df['IsToxic']

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y 
)

# Definición de las etiquetas multi-etiqueta
Y_multi_train = df.iloc[X_train.index][toxic_cols] 
Y_multi_test = df.iloc[X_test.index][toxic_cols]

# Vectorización TF-IDF
stop_words_list = list(ENGLISH_STOP_WORDS) 
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), stop_words=stop_words_list, max_df=0.9, min_df=5 
)

X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

print(f"✅ Vectorización completada. Características: {X_train_vectorized.shape[1]}")

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# =========================================================================
# II. FASE DE ENTRENAMIENTO BASE (Sin optimización)
# =========================================================================
from sklearn.multiclass import OneVsRestClassifier
import lightgbm as lgb 
from sklearn.metrics import f1_score, classification_report


# --- 4. Modelización: Entrenamiento y Evaluación con LightGBM (Base) ---
print("\n--- PASO 4: ENTRENAMIENTO Y EVALUACIÓN (Base) ---")

# Parámetros base: n_estimators=150, scale_pos_weight=4
base_clf = lgb.LGBMClassifier(
    objective='binary', 
    metric='binary_logloss', 
    n_estimators=150,
    learning_rate=0.05, 
    random_state=42, 
    scale_pos_weight=4, 
    n_jobs=-1, 
    verbose=-1
)

model_ovr = OneVsRestClassifier(base_clf)
# Asegúrate de que X_train_vectorized y Y_multi_train estén definidos desde el Bloque I
model_ovr.fit(X_train_vectorized, Y_multi_train) 

Y_pred = model_ovr.predict(X_test_vectorized)
f1_micro = f1_score(Y_multi_test, Y_pred, average='micro', zero_division=0)

print("\n--- RESULTADOS BASE ---")
print(f"Micro F1-Score del Modelo: {f1_micro:.4f}")
print("\nInforme de Clasificación:")
print(classification_report(Y_multi_test, Y_pred, target_names=toxic_cols, zero_division=0))

In [ ]:
# =========================================================================
# II.A OPTIMIZACIÓN DEL MODELO LIGHTGBM CON OPTUNA
# =========================================================================
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, classification_report
import lightgbm as lgb 
import optuna 
import warnings
import numpy as np

# Se asume que X_train_vectorized, Y_multi_train, X_test_vectorized, Y_multi_test, y toxic_cols están definidos.

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_lgbm(trial):
    """Función objetivo para optimizar LightGBM."""
    lgb_params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'n_estimators': trial.suggest_int('n_estimators', 200, 700),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 30, 80),
        'max_depth': trial.suggest_int('max_depth', 8, 20),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'scale_pos_weight': trial.suggest_int('scale_pos_weight', 5, 50), # Rango ampliado
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

    base_clf = lgb.LGBMClassifier(**lgb_params)
    model_ovr = OneVsRestClassifier(base_clf)
    
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model_ovr.fit(X_train_vectorized, Y_multi_train) 

    Y_pred = model_ovr.predict(X_test_vectorized)
    f1_micro = f1_score(Y_multi_test, Y_pred, average='micro', zero_division=0)
    
    return f1_micro

print("\n--- PASO 3.1: OPTIMIZANDO LIGHTGBM CON OPTUNA ---")
study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(objective_lgbm, n_trials=100, show_progress_bar=True) 

print(f"\n✅ LightGBM Optimización completada. Mejor F1: {study_lgbm.best_value:.4f}")
print("Mejores parámetros LightGBM:")
print(study_lgbm.best_params)

# --- Evaluación Final LGBM Optimizado ---
best_params_lgbm = study_lgbm.best_params
best_params_lgbm.update({'objective': 'binary', 'metric': 'binary_logloss', 'random_state': 42, 'n_jobs': -1, 'verbose': -1})

base_clf_lgbm_optimized = lgb.LGBMClassifier(**best_params_lgbm)
model_ovr_lgbm_optimized = OneVsRestClassifier(base_clf_lgbm_optimized)
model_ovr_lgbm_optimized.fit(X_train_vectorized, Y_multi_train) 

Y_pred_lgbm_optimized = model_ovr_lgbm_optimized.predict(X_test_vectorized)
f1_micro_lgbm_optimized = f1_score(Y_multi_test, Y_pred_lgbm_optimized, average='micro', zero_division=0)

print("\n--- RESULTADOS LIGHTGBM OPTIMIZADO ---")
print(f"Micro F1-Score del Modelo: {f1_micro_lgbm_optimized:.4f}")
print("\nInforme de Clasificación:")
print(classification_report(Y_multi_test, Y_pred_lgbm_optimized, target_names=toxic_cols, zero_division=0))